# QCM guidé — Séance 3  
**Gradient, contours et segmentation classique**

## Consignes
- Répondez **directement dans les cellules prévues**.
- Pour chaque question, écrire **une seule lettre** parmi `A`, `B`, `C` ou `D`.
- Ne modifiez pas les fonctions.
- Exécutez les cellules **dans l'ordre**.

## Objectif
Ce mini notebook vérifie que vous avez réellement compris :
- le **gradient** ;
- l'intérêt du **lissage gaussien** ;
- la différence entre **Sobel + seuil** et **Canny** ;
- la différence entre **contour** et **segmentation** ;
- les idées de **Otsu** et **watershed**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
from skimage import data

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["image.cmap"] = "gray"
plt.rcParams["axes.grid"] = False

## Fonctions utilitaires
Ces fonctions reprennent la logique vue dans le notebook de la séance.

In [ ]:
def show_image(img, title="Image", cmap="gray", figsize=(6, 4)):
    plt.figure(figsize=figsize)
    if img.ndim == 2:
        plt.imshow(img, cmap=cmap)
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

def show_images(images, titles, ncols=3, cmap="gray", figsize=(14, 8)):
    n = len(images)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = np.atleast_1d(axes).ravel()

    for ax, img, title in zip(axes, images, titles):
        if img.ndim == 2:
            ax.imshow(img, cmap=cmap)
        else:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(title)
        ax.axis("off")

    for ax in axes[n:]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

def gaussian_blur(image, ksize=5, sigma=1.0):
    return cv2.GaussianBlur(image, (ksize, ksize), sigmaX=sigma)

def compute_sobel(image):
    image_f = image.astype(np.float32)
    gx = cv2.Sobel(image_f, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(image_f, cv2.CV_32F, 0, 1, ksize=3)
    magnitude = np.sqrt(gx**2 + gy**2)
    angle_deg = (np.rad2deg(np.arctan2(gy, gx)) + 180) % 180
    return gx, gy, magnitude, angle_deg

def compute_roberts(image):
    image_f = image.astype(np.float32)
    kx = np.array([[1, 0],
                   [0, -1]], dtype=np.float32)
    ky = np.array([[0, 1],
                   [-1, 0]], dtype=np.float32)
    gx = cv2.filter2D(image_f, cv2.CV_32F, kx)
    gy = cv2.filter2D(image_f, cv2.CV_32F, ky)
    magnitude = np.sqrt(gx**2 + gy**2)
    return gx, gy, magnitude

def compute_prewitt(image):
    image_f = image.astype(np.float32)
    kx = np.array([[-1, 0, 1],
                   [-1, 0, 1],
                   [-1, 0, 1]], dtype=np.float32)
    ky = np.array([[-1, -1, -1],
                   [ 0,  0,  0],
                   [ 1,  1,  1]], dtype=np.float32)
    gx = cv2.filter2D(image_f, cv2.CV_32F, kx)
    gy = cv2.filter2D(image_f, cv2.CV_32F, ky)
    magnitude = np.sqrt(gx**2 + gy**2)
    return gx, gy, magnitude

def normalize_to_uint8(image):
    norm = cv2.normalize(image, None, 0, 255, cv2.NORM_MINMAX)
    return norm.astype(np.uint8)

def threshold_gradient(magnitude, thresh):
    binary = np.zeros_like(magnitude, dtype=np.uint8)
    binary[magnitude >= thresh] = 255
    return binary

def apply_canny(image, low_thresh, high_thresh):
    return cv2.Canny(image, threshold1=low_thresh, threshold2=high_thresh)

def simple_threshold(image, thresh):
    _, binary = cv2.threshold(image, thresh, 255, cv2.THRESH_BINARY)
    return binary

def otsu_threshold(image):
    otsu_value, binary = cv2.threshold(
        image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )
    return otsu_value, binary

def apply_watershed(image):
    blurred = cv2.GaussianBlur(image, (5, 5), 1.0)
    _, thresh = cv2.threshold(
        blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )
    kernel = np.ones((3, 3), np.uint8)
    opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=1)
    sure_bg = cv2.dilate(opening, kernel, iterations=2)
    dist = cv2.distanceTransform(opening, cv2.DIST_L2, 5)
    _, sure_fg = cv2.threshold(dist, 0.4 * dist.max(), 255, 0)
    sure_fg = sure_fg.astype(np.uint8)
    unknown = cv2.subtract(sure_bg, sure_fg)
    num_labels, markers = cv2.connectedComponents(sure_fg)
    markers = markers + 1
    markers[unknown == 255] = 0
    color = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    markers = cv2.watershed(color, markers)
    color_result = color.copy()
    color_result[markers == -1] = [255, 0, 0]
    return color_result, markers, sure_fg, unknown

## Images de travail

In [ ]:
img_camera = data.camera()
img_coins = data.coins()

img_synth = np.zeros((256, 256), dtype=np.uint8)
cv2.rectangle(img_synth, (30, 30), (110, 180), 180, thickness=-1)
cv2.circle(img_synth, (180, 120), 45, 255, thickness=-1)
cv2.line(img_synth, (20, 220), (230, 220), 120, thickness=3)

rng = np.random.default_rng(42)
noise = rng.normal(0, 20, size=img_synth.shape)
img_synth_noisy = np.clip(img_synth.astype(np.float32) + noise, 0, 255).astype(np.uint8)
img_synth_blur = gaussian_blur(img_synth_noisy, ksize=5, sigma=1.2)

## Question 1
Regardez l'image de **magnitude du gradient de Sobel**.

**Question :** Que représentent surtout les zones les plus claires ?

A. Des régions uniformes  
B. Des variations fortes d'intensité  
C. Des objets déjà segmentés  
D. Des zones sans information

In [ ]:
_, _, mag_q1, _ = compute_sobel(img_camera)
show_images(
    [img_camera, normalize_to_uint8(mag_q1)],
    ["Image originale", "Magnitude Sobel"],
    ncols=2,
    figsize=(12, 4)
)

In [ ]:
reponse_q1 = ''  # Remplacer par A, B, C ou D

## Question 2
On compare le gradient sur une image propre, une image bruitée et une image bruitée puis lissée.

**Question :** Pourquoi la troisième image de gradient est-elle plus lisible que la deuxième ?

A. Parce qu'elle contient plus de bruit  
B. Parce que le flou gaussien réduit les faux gradients dus au bruit  
C. Parce que Sobel n'a pas été appliqué  
D. Parce que l'image a été segmentée

In [ ]:
_, _, mag_clean, _ = compute_sobel(img_synth)
_, _, mag_noisy, _ = compute_sobel(img_synth_noisy)
_, _, mag_blur, _ = compute_sobel(img_synth_blur)

show_images(
    [normalize_to_uint8(mag_clean), normalize_to_uint8(mag_noisy), normalize_to_uint8(mag_blur)],
    ["Gradient image propre", "Gradient image bruitée", "Gradient après flou gaussien"],
    ncols=3,
    figsize=(15, 4)
)

In [ ]:
reponse_q2 = ''  # Remplacer par A, B, C ou D

## Question 3
On applique un seuillage simple sur la magnitude de Sobel avec trois seuils.

**Question :** Quel résultat correspond au **seuil le plus bas** ?

A. Celui qui montre le plus de pixels blancs  
B. Celui qui montre le moins de pixels blancs  
C. Celui qui ressemble à une segmentation de régions  
D. Celui qui ne contient aucun contour

In [ ]:
_, _, mag_q3, _ = compute_sobel(img_camera)

thresholds = [40, 80, 120]
images = [normalize_to_uint8(mag_q3)]
titles = ["Magnitude Sobel"]

for th in thresholds:
    images.append(threshold_gradient(mag_q3, th))
    titles.append(f"Sobel + seuil {th}")

show_images(images, titles, ncols=4, figsize=(16, 4))

In [ ]:
reponse_q3 = ''  # Remplacer par A, B, C ou D

## Question 4
On compare **Roberts**, **Prewitt** et **Sobel** sur la même image.

**Question :** Quel opérateur est en général le **plus confortable et le plus stable visuellement** ?

A. Roberts  
B. Prewitt  
C. Sobel  
D. Aucun des trois

In [ ]:
_, _, mag_roberts = compute_roberts(img_camera)
_, _, mag_prewitt = compute_prewitt(img_camera)
_, _, mag_sobel, _ = compute_sobel(img_camera)

show_images(
    [img_camera, normalize_to_uint8(mag_roberts), normalize_to_uint8(mag_prewitt), normalize_to_uint8(mag_sobel)],
    ["Originale", "Roberts", "Prewitt", "Sobel"],
    ncols=4,
    figsize=(16, 4)
)

In [ ]:
reponse_q4 = ''  # Remplacer par A, B, C ou D

## Question 5
On compare **Sobel + seuil simple** et **Canny**.

**Question :** Quelle propriété décrit le mieux le résultat de Canny ici ?

A. Des contours plus fins et plus continus  
B. Une image plus floue  
C. Une segmentation complète des objets  
D. Une suppression totale des contours faibles

In [ ]:
_, _, mag_q5, _ = compute_sobel(img_camera)
sobel_binary = threshold_gradient(mag_q5, 90)
canny_binary = apply_canny(img_camera, 50, 120)

show_images(
    [img_camera, sobel_binary, canny_binary],
    ["Originale", "Sobel + seuil", "Canny"],
    ncols=3,
    figsize=(15, 4)
)

In [ ]:
reponse_q5 = ''  # Remplacer par A, B, C ou D

## Question 6
On fait varier les seuils de Canny.

**Question :** Que se passe-t-il quand les seuils deviennent plus forts ?

A. On détecte plus de détails faibles  
B. Les contours deviennent souvent plus propres, mais certains disparaissent  
C. L'image devient segmentée automatiquement  
D. Le gradient n'est plus calculé

In [ ]:
param_sets = [(30, 80), (50, 120), (80, 180)]

images = [img_camera]
titles = ["Originale"]

for low_t, high_t in param_sets:
    images.append(apply_canny(img_camera, low_t, high_t))
    titles.append(f"Canny low={low_t}, high={high_t}")

show_images(images, titles, ncols=4, figsize=(16, 4))

In [ ]:
reponse_q6 = ''  # Remplacer par A, B, C ou D

## Question 7
On compare trois segmentations de `coins` avec un seuil fixe.

**Question :** Quelle phrase est correcte ?

A. Le résultat change peu quand on change le seuil  
B. Le seuil fixe dépend fortement de la valeur choisie  
C. Les trois images sont des contours, pas des segmentations  
D. Le seuillage fixe choisit automatiquement le meilleur seuil

In [ ]:
thresholds = [80, 110, 140]
images = [img_coins]
titles = ["Image originale"]

for th in thresholds:
    images.append(simple_threshold(img_coins, th))
    titles.append(f"Seuil fixe = {th}")

show_images(images, titles, ncols=4, figsize=(16, 4))

In [ ]:
reponse_q7 = ''  # Remplacer par A, B, C ou D

## Question 8
Voici l'histogramme de l'image `coins`, le seuil choisi par Otsu et la segmentation obtenue.

**Question :** Quel est le rôle principal de la méthode d'Otsu ?

A. Lisser l'image avant gradient  
B. Choisir automatiquement un seuil global  
C. Détecter des contours fins  
D. Construire des marqueurs pour watershed

In [ ]:
otsu_value, otsu_binary = otsu_threshold(img_coins)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(img_coins, cmap="gray")
axes[0].set_title("Image originale")
axes[0].axis("off")

axes[1].hist(img_coins.ravel(), bins=32, color="gray", edgecolor="black")
axes[1].axvline(otsu_value, linestyle="--")
axes[1].set_title(f"Histogramme\nSeuil Otsu = {otsu_value:.1f}")

axes[2].imshow(otsu_binary, cmap="gray")
axes[2].set_title("Segmentation Otsu")
axes[2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
reponse_q8 = ''  # Remplacer par A, B, C ou D

## Question 9
On applique watershed avec marqueurs sur `coins`.

**Question :** Le résultat obtenu correspond surtout à :

A. Une simple carte de gradient  
B. Une segmentation en régions avec frontières  
C. Un flou gaussien  
D. Une image de contours de type Canny

In [ ]:
watershed_result, watershed_markers, sure_fg, unknown_zone = apply_watershed(img_coins)

show_images(
    [img_coins, sure_fg, unknown_zone, watershed_result],
    ["Originale", "Premier plan sûr", "Zone d'incertitude", "Watershed (frontières rouges)"],
    ncols=2,
    figsize=(12, 8)
)

In [ ]:
reponse_q9 = ''  # Remplacer par A, B, C ou D

## Question 10
On compare trois approches sur `coins` : **seuil fixe**, **Otsu** et **watershed**.

**Question :** Quelle affirmation est la plus juste ?

A. Les trois méthodes sont strictement identiques  
B. Watershed change de logique : on passe à une approche par régions  
C. Otsu est un détecteur de contours fins  
D. Le seuil fixe est toujours meilleur qu'Otsu

In [ ]:
fixed_binary = simple_threshold(img_coins, 110)
otsu_value, otsu_binary = otsu_threshold(img_coins)
watershed_result, _, _, _ = apply_watershed(img_coins)

show_images(
    [img_coins, fixed_binary, otsu_binary, watershed_result],
    ["Originale", "Seuil fixe", "Otsu", "Watershed"],
    ncols=4,
    figsize=(18, 4)
)

In [ ]:
reponse_q10 = ''  # Remplacer par A, B, C ou D

## Tableau final des réponses
Ne modifiez pas les noms des variables.

In [ ]:
mes_reponses = {
    "q1": reponse_q1,
    "q2": reponse_q2,
    "q3": reponse_q3,
    "q4": reponse_q4,
    "q5": reponse_q5,
    "q6": reponse_q6,
    "q7": reponse_q7,
    "q8": reponse_q8,
    "q9": reponse_q9,
    "q10": reponse_q10,
}
mes_reponses

## Pour l'enseignant — corrigé rapide
Cette cellule peut rester non exécutée pendant le contrôle.

In [ ]:
corrige = {
    "q1": "B",
    "q2": "B",
    "q3": "A",
    "q4": "C",
    "q5": "A",
    "q6": "B",
    "q7": "B",
    "q8": "B",
    "q9": "B",
    "q10": "B",
}
corrige